# CHURNGUARD AI
**Customer Churn Prediction & Retention Intelligence System**

**Day 1:** Understand → Prepare → Build  
**Day 2:** Improve → Test → Document → Present

**Explanation:** This notebook builds the complete churn-classification workflow required by the project brief.

## DAY 1 — UNDERSTAND → PREPARE → BUILD

### 1. Import libraries
**Explanation:** Import everything needed for data handling, visualization, preprocessing, model training, evaluation, and model saving.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score
)

RANDOM_STATE = 42

### 2. Understand the problem
**Explanation:** Predict whether a customer will churn so the business can identify risky customers before they leave.

- **Target:** `Churn`
- **Type:** Binary classification
- **False negative:** Predicting "stay" when the customer actually leaves

### 3. Load the IBM Telco Customer Churn dataset
**Explanation:** Load the public IBM Telco churn dataset recommended when no official challenge CSV is provided.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_URL)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

### 4. Inspect the dataset
**Explanation:** Check dimensions, data types, missing values, duplicates, and target distribution before cleaning.

In [ ]:
print("Rows, Columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["Churn"].value_counts(dropna=False).to_frame("count"))

### 5. Numerical and categorical summary
**Explanation:** Summarize the dataset to understand value ranges and category distributions.

In [ ]:
display(df.describe(include="all").T)

### 6. Clean the data
**Explanation:** Fix `TotalCharges`, remove duplicate rows, remove the identifier, and handle missing values.

In [ ]:
df = df.copy()

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.drop_duplicates().reset_index(drop=True)

if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

df = df.dropna().reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Remaining missing values:", int(df.isnull().sum().sum()))
print("Remaining duplicates:", int(df.duplicated().sum()))

### 7. EDA — Churn distribution
**Explanation:** Visualize how many customers stayed versus churned.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.show()

churn_rate = df["Churn"].eq("Yes").mean() * 100
print(f"Churn rate: {churn_rate:.2f}%")

### 8. EDA — Contract type vs churn
**Explanation:** Compare churn across contract types.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Customers")
plt.xticks(rotation=15)
plt.show()

display((pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100).round(2))

### 9. EDA — Tenure vs churn
**Explanation:** Check whether customer tenure differs between churners and non-churners.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure vs Churn")
plt.xlabel("Churn")
plt.ylabel("Tenure (months)")
plt.show()

### 10. EDA — Monthly charges vs churn
**Explanation:** Check whether monthly charges differ between churners and non-churners.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

### 11. EDA — Internet service vs churn
**Explanation:** Compare churn across internet-service categories.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="InternetService", hue="Churn")
plt.title("Churn by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Customers")
plt.show()

### 12. EDA — Payment method vs churn
**Explanation:** Compare churn across payment methods.

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x="PaymentMethod", hue="Churn")
plt.title("Churn by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Customers")
plt.xticks(rotation=25)
plt.show()

### 13. Prepare features and target
**Explanation:** Separate input features from the churn target and encode the target as 0/1.

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print("X shape:", X.shape)
print("y shape:", y.shape)
display(y.value_counts().rename(index={0: "Stay", 1: "Churn"}).to_frame("count"))

### 14. Identify numerical and categorical features
**Explanation:** Separate feature types for the preprocessing pipeline.

In [ ]:
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

### 15. Split training and testing data
**Explanation:** Keep 20% of the data unseen for model evaluation while preserving the churn ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### 16. Build the preprocessing pipeline
**Explanation:** Standardize numeric data and one-hot encode categorical data.

In [ ]:
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

### 17. Train Model #1 — Logistic Regression
**Explanation:** Train a simple, interpretable baseline classifier.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, logistic_pred, target_names=["Stay", "Churn"]))

### 18. Train Model #2 — Random Forest
**Explanation:** Train a tree-based classifier that can capture nonlinear customer patterns.

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_pred, target_names=["Stay", "Churn"]))

### 19. Compare baseline models
**Explanation:** Compare accuracy, precision, recall, F1-score, and ROC-AUC; use F1 as the main balanced metric.

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob)
    }

baseline_results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob),
    evaluate_model("Random Forest", y_test, rf_pred, rf_prob)
]).sort_values("F1", ascending=False)

display(baseline_results.round(4))

### 20. Select the better baseline
**Explanation:** Select the stronger baseline using F1-score rather than accuracy alone.

In [ ]:
if f1_score(y_test, logistic_pred) >= f1_score(y_test, rf_pred):
    selected_baseline_name = "Logistic Regression"
    selected_baseline_model = logistic_model
    selected_baseline_pred = logistic_pred
    selected_baseline_prob = logistic_prob
else:
    selected_baseline_name = "Random Forest"
    selected_baseline_model = rf_model
    selected_baseline_pred = rf_pred
    selected_baseline_prob = rf_prob

print("Selected baseline:", selected_baseline_name)

### 21. Confusion matrix
**Explanation:** Inspect true/false positives and negatives for the selected baseline model.

In [ ]:
cm = confusion_matrix(y_test, selected_baseline_pred)

disp = ConfusionMatrixDisplay(cm, display_labels=["Stay", "Churn"])
disp.plot()
plt.title(f"Confusion Matrix — {selected_baseline_name}")
plt.show()

tn, fp, fn, tp = cm.ravel()
print("True Negative:", tn)
print("False Positive:", fp)
print("False Negative:", fn)
print("True Positive:", tp)

## DAY 2 — IMPROVE → TEST → DOCUMENT → PRESENT

### 22. Improve the Random Forest
**Explanation:** Tune a small set of hyperparameters and add class balancing to improve churn detection.

In [ ]:
tuned_rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

param_grid = {
    "classifier__n_estimators": [200],
    "classifier__max_depth": [None, 12],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    tuned_rf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

improved_rf_model = grid_search.best_estimator_
improved_rf_pred = improved_rf_model.predict(X_test)
improved_rf_prob = improved_rf_model.predict_proba(X_test)[:, 1]

print("Best parameters:", grid_search.best_params_)
print(classification_report(y_test, improved_rf_pred, target_names=["Stay", "Churn"]))

### 23. Compare before vs after improvement
**Explanation:** Show whether tuning changed the Random Forest F1-score.

In [ ]:
before_f1 = f1_score(y_test, rf_pred)
after_f1 = f1_score(y_test, improved_rf_pred)

display(pd.DataFrame({
    "Version": ["Baseline Random Forest", "Improved Random Forest"],
    "F1 Score": [before_f1, after_f1]
}).round(4))

print(f"F1 change: {after_f1 - before_f1:+.4f}")

### 24. Compare all models and select the final model
**Explanation:** Choose the highest-F1 model from all trained versions.

In [ ]:
all_results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob),
    evaluate_model("Random Forest", y_test, rf_pred, rf_prob),
    evaluate_model("Improved Random Forest", y_test, improved_rf_pred, improved_rf_prob)
]).sort_values("F1", ascending=False).reset_index(drop=True)

display(all_results.round(4))

final_model_name = all_results.loc[0, "Model"]

if final_model_name == "Logistic Regression":
    final_model, final_pred, final_prob = logistic_model, logistic_pred, logistic_prob
elif final_model_name == "Random Forest":
    final_model, final_pred, final_prob = rf_model, rf_pred, rf_prob
else:
    final_model, final_pred, final_prob = improved_rf_model, improved_rf_pred, improved_rf_prob

print("FINAL MODEL:", final_model_name)

### 25. Final confusion matrix
**Explanation:** Evaluate the final selected model on the unseen test set.

In [ ]:
final_cm = confusion_matrix(y_test, final_pred)

final_disp = ConfusionMatrixDisplay(final_cm, display_labels=["Stay", "Churn"])
final_disp.plot()
plt.title(f"Final Confusion Matrix — {final_model_name}")
plt.show()

print(classification_report(y_test, final_pred, target_names=["Stay", "Churn"]))

### 26. Feature importance
**Explanation:** Extract the strongest features so predictions can be connected to practical churn insights.

In [ ]:
fitted_preprocessor = final_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
classifier = final_model.named_steps["classifier"]

if hasattr(classifier, "feature_importances_"):
    importance_values = classifier.feature_importances_
elif hasattr(classifier, "coef_"):
    importance_values = np.abs(classifier.coef_[0])
else:
    importance_values = np.zeros(len(feature_names))

feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance_values
    })
    .sort_values("Importance", ascending=False)
    .head(15)
)

display(feature_importance)

plt.figure(figsize=(9, 6))
sns.barplot(data=feature_importance, x="Importance", y="Feature")
plt.title(f"Top Churn Features — {final_model_name}")
plt.tight_layout()
plt.show()

### 27. Convert probability into a risk level
**Explanation:** Convert churn probability into LOW, MEDIUM, or HIGH risk for easier business action.

In [ ]:
def churn_risk_level(probability):
    if probability < 0.30:
        return "LOW"
    elif probability < 0.60:
        return "MEDIUM"
    return "HIGH"

for p in [0.15, 0.45, 0.80]:
    print(p, "->", churn_risk_level(p))

### 28. Create the customer prediction function
**Explanation:** Predict churn, churn probability, risk level, and a suggested action for a new customer.

In [ ]:
def predict_customer(customer_data):
    customer_df = pd.DataFrame([customer_data])

    probability = float(final_model.predict_proba(customer_df)[0, 1])
    prediction = int(probability >= 0.50)
    risk = churn_risk_level(probability)

    if risk == "HIGH":
        action = "Prioritize for retention outreach, service review, or targeted offer."
    elif risk == "MEDIUM":
        action = "Monitor and consider proactive engagement."
    else:
        action = "Continue normal engagement."

    return {
        "Prediction": "Likely to churn" if prediction == 1 else "Likely to stay",
        "Churn Probability": round(probability, 4),
        "Risk Level": risk,
        "Suggested Action": action
    }

### 29. Test with a sample customer
**Explanation:** Demonstrate the complete system on one new customer record.

In [ ]:
sample_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 5,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 89.50,
    "TotalCharges": 447.50
}

result = predict_customer(sample_customer)
display(pd.DataFrame([result]))

### 30. Save the trained model and result files
**Explanation:** Save the final pipeline and the key result tables for reuse and submission.

In [ ]:
joblib.dump(final_model, "churnguard_model.joblib")
all_results.to_csv("model_comparison.csv", index=False)
feature_importance.to_csv("feature_importance.csv", index=False)

print("Saved: churnguard_model.joblib")
print("Saved: model_comparison.csv")
print("Saved: feature_importance.csv")

### 31. Final project summary
**Explanation:** Print the required final metrics, important features, business recommendation, and limitation.

In [ ]:
final_accuracy = accuracy_score(y_test, final_pred)
final_precision = precision_score(y_test, final_pred, zero_division=0)
final_recall = recall_score(y_test, final_pred, zero_division=0)
final_f1 = f1_score(y_test, final_pred, zero_division=0)
final_auc = roc_auc_score(y_test, final_prob)

print("=" * 60)
print("CHURNGUARD AI — FINAL RESULTS")
print("=" * 60)
print("Dataset: IBM Telco Customer Churn")
print("Number of records:", len(df))
print("Models tested: Logistic Regression, Random Forest, Improved Random Forest")
print("Selected model:", final_model_name)
print(f"Accuracy:  {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall:    {final_recall:.4f}")
print(f"F1 Score:  {final_f1:.4f}")
print(f"ROC-AUC:   {final_auc:.4f}")

print("\nTop important features:")
for feature in feature_importance["Feature"].head(5):
    print("-", feature)

print("\nBusiness recommendation:")
print("Prioritize HIGH-risk customers for proactive retention action.")

print("\nLimitation:")
print("Predictions indicate statistical risk; they do not guarantee that an individual customer will churn.")

## OPTIONAL — Streamlit Web App
**Explanation:** The core project is complete; use the included `app.py` only if you want the optional web interface.